In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection


In [ ]:
import numpy as np
BASE_DIR = "./dataset_windows20/"

X_train = np.load(BASE_DIR + "train/X.npy")
y_train = np.load(BASE_DIR + "train/y_bin.npy")
y_ch_train = np.load(BASE_DIR + "train/y_ch.npy")

X_val = np.load(BASE_DIR + "val/X.npy")
y_val = np.load(BASE_DIR + "val/y_bin.npy")
y_ch_val = np.load(BASE_DIR + "val/y_ch.npy")

X_test = np.load(BASE_DIR + "test/X.npy")
y_test = np.load(BASE_DIR + "test/y_bin.npy")
y_ch_test = np.load(BASE_DIR + "test/y_ch.npy")

In [ ]:

print(y_ch_train.shape)

(279986, 15)


In [ ]:
X_train = X_train[:, :, 1:]
X_val   = X_val[:, :, 1:]
X_test  = X_test[:, :, 1:]

print(X_train.shape)


(279986, 20, 16)


In [ ]:
X_train = X_train[:, :, 0:-1]
X_val   = X_val[:, :, 0:-1]
X_test  = X_test[:, :, 0:-1]
print(X_train.shape)

(279986, 20, 15)


In [ ]:
## Preprocessing

In [ ]:
X_train_nom = X_train[y_train == 0]   # solo nominal

mean_feat = X_train_nom.mean(axis=(0,1))   # (F,)
std_feat  = X_train_nom.std(axis=(0,1)) + 1e-8


In [ ]:
def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_train_s = scale_windows(X_train, mean_feat, std_feat)
X_val_s   = scale_windows(X_val,   mean_feat, std_feat)
X_test_s  = scale_windows(X_test,  mean_feat, std_feat)


In [ ]:
import pywt
import numpy as np

def compute_dwt_windows(X, wavelet="db4", level=1):
    """
    X: (N_windows, window_size, n_channels) o (N_windows, window_size)
    return:
        A: Approximation coefficients (low-frequency)
        D: Detail coefficients (high-frequency, concatenated)
    """
    if X.ndim == 2:
        X = X[..., np.newaxis]

    N, W, F = X.shape

    A_list = []
    D_list = []

    for i in range(N):
        A_ch = []
        D_ch = []

        for ch in range(F):
            coeffs = pywt.wavedec(X[i, :, ch], wavelet, level=level)

            A = coeffs[0]                  # Approximation
            D = np.concatenate(coeffs[1:]) # All detail levels

            A_ch.append(A)
            D_ch.append(D)

        A_list.append(np.stack(A_ch, axis=1))
        D_list.append(np.stack(D_ch, axis=1))

    return np.array(A_list), np.array(D_list)
# A_windows, D_windows = compute_dwt_windows(X)
# print("A:", A_windows.shape)
# print("D:", D_windows.shape)
def adaptive_threshold_detector_HF(window, thresholds):
    win_max = np.max(window, axis=0)
    return int(np.any(win_max > thresholds))
def adaptive_threshold_detector_LF(window, thresholds):
    win_min = np.min(window, axis=0)
    return int(np.any(win_min < thresholds))
def compute_thresholds(feature_windows, y_bin, k=2.0, mode="high"):
    # feature_windows: (N, coeffs, F)
    nominal = feature_windows[y_bin == 0]

    vals = np.max(nominal, axis=1) if mode == "high" else np.min(nominal, axis=1)

    mu = np.mean(vals, axis=0)
    sigma = np.std(vals, axis=0)

    if mode == "high":
        return mu + k * sigma
    else:
        return mu - k * sigma
# TH_D = compute_thresholds(D_windows, y_bin, mode="high")  # spikes
# TH_A = compute_thresholds(A_windows, y_bin, mode="low")   # stuck
# y_pred_D = np.array([
#     adaptive_threshold_detector_HF(w, TH_D)
#     for w in D_windows
# ])

# y_pred_A = np.array([
#     adaptive_threshold_detector_LF(w, TH_A)
#     for w in A_windows
# ])



In [ ]:
A_train, D_train = compute_dwt_windows(X_train_s)
A_val,   D_val   = compute_dwt_windows(X_val_s)
A_test,  D_test  = compute_dwt_windows(X_test_s)


In [ ]:
print(D_train.shape)
# (N_samples, n_coeffs, n_channels)


(279986, 13, 15)


In [ ]:
mean = np.mean(D_train, axis=(0,1), keepdims=True)
std  = np.std(D_train, axis=(0,1), keepdims=True) + 1e-8

D_train_n = (D_train - mean) / std
D_val_n   = (D_val   - mean) / std
D_test_n  = (D_test  - mean) / std


In [ ]:
def compute_energy_features(D):
    return np.sum(D**2, axis=1)

E_train = compute_energy_features(D_train_n)
E_val   = compute_energy_features(D_val_n)
E_test  = compute_energy_features(D_test_n)

print(E_train.shape)
# (N, 15)

(279986, 15)


In [ ]:
print(y_ch_train.shape)

(279986, 15)


In [ ]:
from sklearn.linear_model import LogisticRegression

models = []

for ch in range(15):

    clf = LogisticRegression(
        max_iter=1000,
        class_weight='balanced'  # MUY IMPORTANTE
    )

    clf.fit(E_train, y_ch_train[:, ch])

    models.append(clf)

In [ ]:
y_pred = np.zeros_like(y_ch_val)

for ch in range(15):

    prob = models[ch].predict_proba(E_val)[:,1]

    y_pred[:,ch] = (prob > 0.7).astype(int)   # threshold ajustable

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_ch_val,
    y_pred,
    target_names=[f"ch{i}" for i in range(15)],
    digits=4
))

              precision    recall  f1-score   support

         ch0     0.6602    0.7239    0.6906      2727
         ch1     0.7114    0.7079    0.7096      2636
         ch2     0.7657    0.7241    0.7443      2370
         ch3     0.7297    0.7294    0.7295      3038
         ch4     0.9022    0.6051    0.7244      2317
         ch5     0.8213    0.6500    0.7257      2737
         ch6     0.8470    0.5952    0.6991      2883
         ch7     0.8012    0.4704    0.5928      2570
         ch8     0.8182    0.5190    0.6351      2559
         ch9     0.9162    0.6608    0.7678      2745
        ch10     0.9338    0.5910    0.7239      3125
        ch11     0.9353    0.6897    0.7940      2559
        ch12     0.9982    0.8780    0.9343      2591
        ch13     0.9995    0.8583    0.9236      2492
        ch14     0.9910    0.7787    0.8721      2268

   micro avg     0.8427    0.6768    0.7507     39617
   macro avg     0.8554    0.6788    0.7511     39617
weighted avg     0.8530   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# TEST

In [ ]:
y_pred = np.zeros_like(y_ch_test)

for ch in range(15):

    prob = models[ch].predict_proba(E_test)[:,1]

    y_pred[:,ch] = (prob > 0.7).astype(int)   # threshold ajustable

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_ch_test,
    y_pred,
    target_names=[f"ch{i}" for i in range(15)],
    digits=4
))

              precision    recall  f1-score   support

         ch0     0.5702    0.7069    0.6312      2286
         ch1     0.6425    0.7661    0.6989      2330
         ch2     0.7680    0.6514    0.7049      2582
         ch3     0.7039    0.7189    0.7113      2629
         ch4     0.9086    0.6732    0.7733      2362
         ch5     0.8193    0.6956    0.7524      2704
         ch6     0.8671    0.6201    0.7231      3072
         ch7     0.8612    0.6229    0.7229      2710
         ch8     0.8694    0.5988    0.7092      2869
         ch9     0.9050    0.6159    0.7330      3015
        ch10     0.9401    0.5906    0.7254      2633
        ch11     0.9479    0.6303    0.7571      2975
        ch12     0.9974    0.7198    0.8362      3166
        ch13     0.9991    0.8026    0.8901      2670
        ch14     0.9646    0.7344    0.8339      2150

   micro avg     0.8342    0.6735    0.7453     40153
   macro avg     0.8510    0.6765    0.7469     40153
weighted avg     0.8570   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
